In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, brier_score_loss, classification_report, confusion_matrix
from sklearn.calibration import calibration_curve

## Import First 24-Hour Dataframe

In [ ]:
df_24hour = pd.read_pickle('micu_24hours.pkl')

# Check shape of dataframe
print("Baseline Data Shape:", df_24hour.shape)

## Split Data into Train, Validation, and Test Sets

In [ ]:
# Separate features and target
X = df_24hour.drop(columns=['stay_id', 'los', 'extended_stay'])
y = df_24hour['extended_stay']

# Save feature names for intepretation
feature_names = X.columns

# Establish test set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)

# Establish train and validation sets from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.1111, random_state=42, stratify=y_temp)

## Data Transformation and Preprocessing

In [ ]:
# Establish numeric vs. categorical columns
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Build Numeric Transformer
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Build Categorical Transformer
cat_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

# Combine both transformers into preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Fit train set, transform train, val, and test sets
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

## Create & Train Logistic Regression Model

In [ ]:
log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1500, random_state=42))
])
log_reg.fit(X_train_processed, y_train)

## Evaluate Performance on Validation Set

### Generate Validation Predictions

In [ ]:
y_val_pred = log_reg.predict(X_val_processed)
y_val_prob = log_reg.predict_proba(X_val_processed)[:, 1]

### Validation ROC-AUC and AUPRC

In [ ]:
val_auc = roc_auc_score(y_val, y_val_prob)
val_auprc = average_precision_score(y_val, y_val_prob)

print(f"Validation ROC-AUC: {val_auc:.3f}")
print(f"Validation AUPRC: {val_auprc:.3f}")

#### Given that extended stays make up 25% of the data, **relative AUPRC improvement over baseline is 70%**.

### Validation ROC Curve

In [ ]:
# Calculate validation false positive rate, true positive rate, and thresholds
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, y_val_prob)

# Plot the validation ROC curve
plt.figure(figsize=(7, 5))
plt.plot(fpr_val, tpr_val, label=f'Validation ROC (AUC = {val_auc:.3f})')

# Plot the random guess baseline
plt.plot([0, 1], [0, 1], label="Random Guess (AUC = 0.500)")

#Format plot
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation Set Receiver Operating Characteristic (ROC)")
plt.legend(loc='lower right')
plt.show()

### Validation Feature Importance

In [ ]:
# Extract coefficients
log_reg_step = log_reg.named_steps['lr']
coefficients = np.squeeze(log_reg_step.coef_)

# Extract new categorical feature names
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
encoded_cat_features = cat_encoder.get_feature_names_out(cat_features).tolist()

# Combine original numeric names with new one-hot encoded names
all_feature_names = num_features + encoded_cat_features

# Create dataframe mapping features to their weights
importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefficients,
    'Absolute_Coefficient': np.abs(coefficients)
}).sort_values(by='Absolute_Coefficient', ascending=False)

print(importance_df.to_string(index=False))

### Validation Brier Score

In [ ]:
val_brier = brier_score_loss(y_val, y_val_prob)
print(f"Validation Brier Score Loss: {val_brier:.3f}")

#### Given that extended stays make up 25% of the data, **relative Brier score improvement over baseline is 11.5%**.

### Validation Calibration Curve (Reliability Diagram)

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "How to best format a calibration curve in python matplotlib?"
#### Usage: Used aspects of recommended formatting approach for graph.
#### -------------------------------------------------------------------------

In [ ]:
# Compute calibration curve data
prob_true_lr_val, prob_pred_lr_val = calibration_curve(y_val, y_val_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lr_val, prob_true_lr_val, 's-', label=f"Logistic Regression (Brier: {val_brier:.3f})")

# Format plot
plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("Logistic Regression Calibration Curve (Validation Set)")
plt.legend(loc='lower right')
plt.show()

## Hyperparameter Tuning with Grid Search

In [ ]:
import warnings
# Silence warnings
warnings.filterwarnings('ignore', category=FutureWarning, message=".*'penalty' was deprecated.*")

# Create pipeline to scale data dynamically for each cross-validation fold
cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, random_state=42))
])

# Define hyperparameters to search through
param_grid = {
    'lr__l1_ratio': [0.0, 1.0],
    'lr__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'lr__max_iter': [1500]
}

# Set up Grid Search
grid_search = GridSearchCV(
    estimator=cv_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Run search on training data
grid_search.fit(X_train_processed, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print(f"Best Cross-Validation ROC-AUC: {grid_search.best_score_:.4f}")

### Check scores of lower C values

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "How can I review grid search result scores to confirm the 
#### best hyperparameter tuning?"
#### Usage: Used 'leaderboard' approach to compare C values.
#### -------------------------------------------------------------------------

In [ ]:
# Convert the grid search results into a clean dataframe
results_df = pd.DataFrame(grid_search.cv_results_)

# Filter down to the most important columns and sort by performance
leaderboard = results_df[['param_lr__C', 'mean_test_score', 'std_test_score']]
leaderboard = leaderboard.sort_values(by='mean_test_score', ascending=False)

print("Hyperparameter Tuning Leaderboard")
print(leaderboard.to_string(index=False))

#### C=100 and C=1.0 have essentially the same predictive power. Proceeding with C=1.0 ensures more stability on unseen clinical data.

## Create & Train Final Logistic Regression Model with Optimal Hyperparameters

In [ ]:
log_reg_final = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        C=1.0,
        l1_ratio=0.0,
        max_iter=1500,
        random_state=42
    ))
])

log_reg_final.fit(X_train_processed, y_train)

### Regenerate Validation Predictions for J Score Threshold Calculation

In [ ]:
y_val_prob2 = log_reg_final.predict_proba(X_val_processed)[:, 1]

### Validation Youden's J Score

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "How is Youden's J score used to evaluate machine learning performance 
#### in python?"
#### Usage: Used best J value approach and explanation for interpretation.
#### -------------------------------------------------------------------------

In [ ]:
# Calculate validation false positive rate, true positive rate, and thresholds
fpr_val2, tpr_val2, thresholds_val2 = roc_curve(y_val, y_val_prob2)

val_j = tpr_val2 - fpr_val2

# Find the index of the highest J score
best_idx_val = np.argmax(val_j)
best_threshold_val = thresholds_val[best_idx_val]
best_j_value_val = val_j[best_idx_val]

print(f"Validation Optimal J Score Threshold: {best_threshold_val:.4f}")
print(f"Validation Maximized Youden's J Score: {best_j_value_val:.4f}")

## Evaluate Performance on Test Set

### Generate Test Predictions (Implement Optimal J Score Threshold)

In [ ]:
y_test_prob = log_reg_final.predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_prob >= best_threshold_val).astype(int)

### Test Metrics

In [ ]:
# Calculate test false positive rate, true positive rate, and thresholds
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_prob)

# Calculate final test metrics
test_roc = roc_auc_score(y_test, y_test_prob)
test_auprc = average_precision_score(y_test, y_test_prob)
test_brier = brier_score_loss(y_test, y_test_prob)

print(f"Final Test ROC-AUC: {test_roc:.4f}")
print(f"Final Test AUPRC: {test_auprc:.4f}")
print(f"Final Test Brier Loss: {test_brier:.4f}")

print("Final Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Standard Stay", "Extended Stay"]))

print("Final Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

#### **Final relative AUPRC improvement over baseline is 72%**.

#### **Final relative Brier score improvement over baseline is 12%**.

### Test Calibration Curve (Reliability Diagram) 

In [ ]:
# Compute calibration curve data
prob_true_lr_test, prob_pred_lr_test = calibration_curve(y_test, y_test_prob, n_bins=10, strategy='quantile')

# Plot curve
plt.figure(figsize=(7, 7))
plt.plot([0,1], [0,1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_lr_test, prob_true_lr_test, 's-', label=f"Logistic Regression (Brier: {test_brier:.3f})")

# Format plot
plt.xlabel("Predicted Probability of Extended Stay")
plt.ylabel("Actual Extended Stay Rate")
plt.title("Logistic Regression Calibration Curve (Test Set)")
plt.legend(loc='lower right')
plt.show()